- [gpt 5.6 cutoff](https://developers.openai.com/api/docs/models/gpt-5.6-luna) 
    - Feb 16, 2026 knowledge cutoff

In [1]:
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

True

### GPT에 인터넷 검색 기능 추가하기

In [2]:
model = ChatOpenAI(model="gpt-5.6-luna")
response = model.invoke("최근 한국 아이돌 키키가 발표한 신곡은 무엇인가요?")
print(response.content)

한국 아이돌 그룹 **키키(KiiiKiii)**가 최근 발표한 신곡은 **〈DANCING ALONE〉**입니다. 2025년 6월 공개된 디지털 싱글로, 키키 특유의 밝고 몽환적인 분위기가 담긴 곡입니다.


In [3]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke("최근 한국 아이돌 키키가 발표한 신곡은 무엇인가요?")

print(docs)

C:\Users\user\AppData\Local\Temp\ipykernel_25616\1046302797.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


snippet: August 10, 2026 - 한눈에 보는 오늘 : 방송/가요 - 뉴스 : [동아닷컴] [동아닷컴 정희연 기자] 특유의 키치한 매력으로 사랑 받아온 그룹 키키(KiiiKiii)가 이번에도 유쾌하고 위트 넘치는 느낌으로 중무장했다. 타이틀곡은 마치 쨍한 여름 햇살이 ..., title: 키키가 말아주는 Y2K는 다르다…쨍한 여름의 'Pop Off Pop Off' (종합)[DA신곡] : 네이트 연예, link: https://m.news.nate.com/view/20260810n27444;
snippet: August 10, 2026 - 앞서 '404(New Era)'로 음원차트를 강타했던 이들은 또 한 번 막강한 매력과 중독성을 지닌 곡을 내놨다.키키(지유, 이솔, 수이, 하음, 키야)는 10일, title: 서머송도 키키가 하면 다르다…신나고 즐겁고 힙하기까지 '팝 오프 팝 오프' [신곡in가요] : 네이트 연예, link: https://m.news.nate.com/view/20260810n27451;
snippet: August 11, 2026 - 11일 소속사 스타쉽엔터테인먼트에 따르면 키키(지유 이솔 수이 하음 키야)가 10일 발표한 세 번째 미니앨범 'WhyKiiiKiii(와이키키)' 타이틀곡 'Pop Off Pop Off(팝 오프 팝 오프)'는 11일 자 유튜브 ..., title: 키키 "새로운 매력 가득한 'WhyKiiiKiii'…많이 들려주고 싶어"(일문일답) : 네이트 연예, link: https://m.news.nate.com/view/20260811n28973;
snippet: August 10, 2026 - (서울=연합뉴스) 김선우 기자 = 그룹 키키가 10일 오후 6시 미니 3집 '와이키키'(WhyKiiiKiii)를 발매한다고 스타쉽엔터테인먼트가 밝혔다., title: "왜 키키인가"…키키, 오늘 미니앨범 '와이키키' 발매 | 연합뉴스, link: https://www.yna.co.kr/amp/view/AKR20260

#### 질의 확장 구현

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | model

In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = InMemoryChatMessageHistory()
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("최근 한국 아이돌 키키가 발표한 신곡은 무엇인가요?") 

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer.content)

키키(KiiiKiii)가 최근 발표한 신곡은 **‘Pop Off Pop Off(팝 오프 팝 오프)’**입니다.  

이 곡은 2026년 8월 10일 발매된 세 번째 미니앨범 **《WhyKiiiKiii(와이키키)》**의 타이틀곡으로, 경쾌하고 중독성 강한 여름 분위기의 곡입니다.


### 검색 기능에 옵션 설정하기
- API 래퍼를 사용하여 검색 옵션 설정

In [6]:
# DuckDuckGo API wrapper를 사용하여 검색할 때 검색 매개변수를 설정하기 위한 클래스 import
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 한국 지역("kr-kr")을 기준, 최근 일주일("w") 내의 검색 결과를 가져오도록 초기화
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")

In [7]:
# 검색 기능을 위한 DuckDuckGoSearchResults 초기화
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,      # 앞에서 정의한 API wrapper를 사용
    source="news",            # 뉴스 소스에서만 검색하도록 지정
    results_separator=';\n'   # 결과 항목 사이에 구분자 사용 (세미콜론과 줄바꿈)
)

In [8]:
# "리센느"를 검색하고 결과를 docs에 저장
docs = search.invoke("리센느")

# 검색 결과 출력
print(docs)

snippet: RESCENE （韓語： 리센느， 羅馬化：ri sen neu； 日語：リセンヌ）是 韓國 THE MUZE娛樂於2024年推出的女子團體，由Woni、Liv、May、Zena、Minami組成 [1][2]。2024年3月26日發行首張單曲專輯《Re:Scene》正式出道，首週銷量達34,125張，位居K-pop女團出道專輯首週銷量第10名 [3][4]。團體概念源於其團名，意指「透過 ..., title: Rescene - 维基百科，自由的百科全书, link: https://zh.wikipedia.org/wiki/RESCENE;
snippet: 9.18 (금) 서울대학교 축제 리센느 공연 영상입니다~ 즐감하세요!!저도 누군가의 피디니무가 되는 날까지 최선을 다하겠습니다.#리센느 #서울대 #축제 ..., title: 내가 보고 싶어서 만든 서울대 축제 리센느 모음 Zip., link: https://m.youtube.com/watch?v=EpXMYWhg0NY;
snippet: 2026 07 08 Pretty Girl - Special Single COME BACK - 리센느 갤러리에 다양한 이야기를 남겨주세요., title: 260918 공트 리센느 컴백확정 - 리센느 마이너 갤러리, link: https://gall.dcinside.com/mgallery/board/view/?id=rescene1&no=511030;
snippet: 일본 유튜버들이 줄줄이 리센느 영상 올리는 이유 | 나고야 아시안게임 홍보대사 리센느 일본 반응 슈탐정 16.3K subscribers Subscribe, title: 일본 유튜버들이 줄줄이 리센느 영상 올리는 이유 | 나고야 아시안게임 홍보대사 리센느 일본 반응, link: https://m.youtube.com/watch?v=zX6gOnw1hgQ


In [9]:
# DuckDuckGo를 이용해 starnewskorea.com 사이트에서 리센느에 대한 뉴스 검색
docs = search.invoke("site:starnewskorea.com 리센느")
docs

'snippet: 2 days ago · 걸그룹 리센느(RESCENE)가 오는 11월 컴백 활동에 나선다. 소속사 더뮤즈엔터테인먼트는 18일 스타뉴스에 "리센느가 오는 11월 3일 새 앨범을 발표한다"고 밝혔다. 리센느는 이로써 지난 7월 발표한 \'프리티 걸\' 이후 약 4개월 만에 컴백하게 됐다. 리메이크가 아닌 신곡으로는 지난 4월 발표한 \'런어웨이 ..., title: [공식]\'대세\' 리센느 11월 3일 컴백 확정..역주행 이어 연말 접수, link: https://www.starnewskorea.com/music/2026/09/18/2026091816364625648;\nsnippet: 6 days ago · 걸 그룹 리센느(RESCENE)가 장기 흥행 중이다. 리센느(원이, 리브, 미나미, 메이, 제나)는 미니 1집 \'SCENEDROME\'(씬드롬)의 타이틀곡 \'LOVE ATTACK\'(러브 어택)으로 멜론 TOP100 1위를 장기간 유지하고 있는 가운데 리메이크 싱글 \'Pretty Girl\'(프리티 걸)까지 2위(지난 13일 오후 11시 기준)에 오르며 두 곡을 나란히 ..., title: 리센느, 역주행·정주행 다 잡았다..멜론 톱100 1·2위 달성, link: https://www.starnewskorea.com/music/2026/09/14/2026091414025971047;\nsnippet: 4 days ago · 그룹 리센느 미나미가 \'대세\' 임을 입증했다. 16일 방송된 MBC 예능 프로그램 \'라디오스타\'는 \'나고야-호! 아시안게임\' 특집으로 꾸며져 이형택, 이혜정, 조준호, 미나미가 출연했다., title: 리센느 미나미, 대세 입증 "광고 문의만 100개..잘 시간도 부족" [라..., link: https://www.starnewskorea.com/broadcast-show/2026/09/16/2026091622565821277;\nsnippet: 4 days ago · 아시안게임 \' 특집으로 꾸며진다.

### 기사 링크 가져오기

In [10]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄바꿈 기준으로 분리하고, 각 결과 항목에서 링크를 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목을 출력하여 확인
    link = doc.split("link:")[1].strip()  # 각 항목에서 'link:' 이후의 URL 부분만 추출
    links.append(link)  # 추출한 링크를 리스트에 추가

# 모든 링크를 출력
print(links)

snippet: 2 days ago · 걸그룹 리센느(RESCENE)가 오는 11월 컴백 활동에 나선다. 소속사 더뮤즈엔터테인먼트는 18일 스타뉴스에 "리센느가 오는 11월 3일 새 앨범을 발표한다"고 밝혔다. 리센느는 이로써 지난 7월 발표한 '프리티 걸' 이후 약 4개월 만에 컴백하게 됐다. 리메이크가 아닌 신곡으로는 지난 4월 발표한 '런어웨이 ..., title: [공식]'대세' 리센느 11월 3일 컴백 확정..역주행 이어 연말 접수, link: https://www.starnewskorea.com/music/2026/09/18/2026091816364625648
snippet: 6 days ago · 걸 그룹 리센느(RESCENE)가 장기 흥행 중이다. 리센느(원이, 리브, 미나미, 메이, 제나)는 미니 1집 'SCENEDROME'(씬드롬)의 타이틀곡 'LOVE ATTACK'(러브 어택)으로 멜론 TOP100 1위를 장기간 유지하고 있는 가운데 리메이크 싱글 'Pretty Girl'(프리티 걸)까지 2위(지난 13일 오후 11시 기준)에 오르며 두 곡을 나란히 ..., title: 리센느, 역주행·정주행 다 잡았다..멜론 톱100 1·2위 달성, link: https://www.starnewskorea.com/music/2026/09/14/2026091414025971047
snippet: 4 days ago · 그룹 리센느 미나미가 '대세' 임을 입증했다. 16일 방송된 MBC 예능 프로그램 '라디오스타'는 '나고야-호! 아시안게임' 특집으로 꾸며져 이형택, 이혜정, 조준호, 미나미가 출연했다., title: 리센느 미나미, 대세 입증 "광고 문의만 100개..잘 시간도 부족" [라..., link: https://www.starnewskorea.com/broadcast-show/2026/09/16/2026091622565821277
snippet: 4 days ago · 아시안게임 ' 특집으로 꾸며진다. 미나미는 '러브어택' 역주행 후 리센느 멤버

In [11]:
# Langchain의 WebBaseLoader를 사용하여 웹 페이지의 내용을 불러옵니다.
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 객체를 생성. 'links'는 웹 페이지의 URL 목록을 담고 있는 변수
# bs_get_text_kwargs는 BeautifulSoup의 get_text() 메소드에 전달될 추가 인자
loader = WebBaseLoader(
    web_paths=links,  # 웹 페이지의 링크 목록을 지정
    bs_get_text_kwargs={
        "strip": True  # 웹 페이지에서 텍스트를 가져올 때 앞뒤의 공백을 제거
    },
)

# 비동기로 웹 페이지의 내용을 로드하고, 각 문서를 page_contents 리스트에 추가
page_contents = []  # 각 웹 페이지의 내용을 저장할 리스트입니다.
async for doc in loader.alazy_load():
    page_contents.append(doc)  # 불러온 문서를 page_contents 리스트에 추가

# page_contents에 있는 각 웹 페이지의 내용을 출력
for content in page_contents:
    print(content)  # 웹 페이지의 내용을 출력
    print('--------------')  # 페이지 구분을 위해 구분선을 출력

USER_AGENT environment variable not set, consider setting it to identify your requests.
d:\GitHub\learn_do_it_llm_agent\ch10\.venv\Lib\site-packages\langchain_community\document_loaders\web_base.py:299: UserWarning: For better logging of progress, `pip install tqdm`
  warnings.warn("For better logging of progress, `pip install tqdm`")


page_content='[공식]'대세' 리센느 11월 3일 컴백 확정..역주행 이어 연말 접수 | 스타뉴스KO연예스포츠비즈/라이프최신 뉴스이슈/연재영상/포토스타랭킹연예K-POP연예K-POP[공식]'대세' 리센느 11월 3일 컴백 확정..역주행 이어 연말 접수발행:2026.09.18 ・ 16:42조회수:윤상근 기자Google 검색 선호 출처로 추가걸그룹 리센느 /사진=(고양=뉴스1) 권현진 기자걸그룹 리센느(RESCENE)가 오는 11월 컴백 활동에 나선다.소속사 더뮤즈엔터테인먼트는 18일 스타뉴스에 "리센느가 오는 11월 3일 새 앨범을 발표한다"고 밝혔다.리센느는 이로써 지난 7월 발표한 '프리티 걸' 이후 약 4개월 만에 컴백하게 됐다. 리메이크가 아닌 신곡으로는 지난 4월 발표한 '런어웨이' 이후 약 7개월 만이다.리센느는 멤버 원이의유튜브'안녕하세요원이입니다잘부탁드립니다'의 화제성 속에 올해 대세로 떠올랐다.이후 '러브 어택'이 약 2년 만에 음원차트 정상까지 오르는 역주행을 기록하기도 했다.Google 검색에서 스타뉴스를 더 자주 만나보세요.<저작권자 © 스타뉴스, 무단전재 및 재배포 금지>브리핑추천 기사윤상근 기자기자홈좋아요연예-K-POP의 인기 급상승 뉴스연예-K-POP의 최신 뉴스AD앱 다운받기STARNEWS APPSTARPOLL회사소개개인정보처리방침청소년보호정책고충처리인저작권규약이용약관RSS팩트체크보도윤리강령(주)브릴리언트코리아머니투데이MTN뉴시스news1지디넷시대thebellAAA아이즈(ize)스타폴스타뉴스 사업자 정보주소: 서울시 종로구 청계천로 11(서린동, 청계한국빌딩)발행인/편집인: 박준철청소년 보호책임자: 문완식등록번호:서울 아01055등록일:2009.12.10제호:스타뉴스발행일:2009.12.10전화번호: 02-767-6843ㆍ02-724-0985주소: 서울시 종로구 청계천로 11(서린동, 청계한국빌딩)발행인/편집인: 박준철청소년 보호책임자: 문완식등록번호:서울 아01055등록일:2009.12.10제호:스타뉴스발행일:2009.12

위처럼 웹 페이지의 거의 텍스트를 가져오면 기사 내용과 상관없는 내용까지 포함되어

뷰티풀 수프를 이용해 특정 영역만 가져오도록 한다.

In [12]:
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup


# 주어진 URL에서 기사 텍스트를 가져오는 함수
def get_article_text(url):
    try:
        # URL에 GET 요청을 보냄
        response = requests.get(url)
        # 요청이 성공하지 못하면 예외를 발생시킴
        response.raise_for_status()

        # BeautifulSoup을 사용하여 HTML 내용을 파싱
        soup = BeautifulSoup(response.content, 'html.parser')

        domain = urlparse(url).netloc
        article = None

        # 스타뉴스
        if "starnewskorea.com" in domain:
            article = soup.find("div", id="article-body")

        # YTN
        elif "ytn.co.kr" in domain:
            article = soup.find("article", class_="story-news article")

        # 기사를 찾았다면 그 텍스트를 반환
        if article:
            return article.get_text(" ", strip=True)

        return "기사 내용을 찾을 수 없습니다."

    # 요청이 실패할 경우 예외 처리
    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"

In [13]:
# URL 목록의 각 링크를 반복하면서 기사 텍스트를 출력
articles = []    # 가져온 내용을 리스트에 담기 위한 변수 선언
for link in links:
    print(f"Input: {link}\n")
    article_text = get_article_text(link)
    print(f"Content:\n{article_text}")
    print("--------------------------------------------------")
    articles.append(article_text)

Input: https://www.starnewskorea.com/music/2026/09/18/2026091816364625648

Content:
걸그룹 리센느 /사진=(고양=뉴스1) 권현진 기자 걸그룹 리센느(RESCENE)가 오는 11월 컴백 활동에 나선다. 소속사 더뮤즈엔터테인먼트는 18일 스타뉴스에 "리센느가 오는 11월 3일 새 앨범을 발표한다"고 밝혔다. 리센느는 이로써 지난 7월 발표한 '프리티 걸' 이후 약 4개월 만에 컴백하게 됐다. 리메이크가 아닌 신곡으로는 지난 4월 발표한 '런어웨이' 이후 약 7개월 만이다. 리센느는 멤버 원이의 유튜브 '안녕하세요원이입니다잘부탁드립니다'의 화제성 속에 올해 대세로 떠올랐다. 이후 '러브 어택'이 약 2년 만에 음원차트 정상까지 오르는 역주행을 기록하기도 했다. <저작권자 © 스타뉴스, 무단전재 및 재배포 금지>
--------------------------------------------------
Input: https://www.starnewskorea.com/music/2026/09/14/2026091414025971047

Content:
/사진=더뮤즈엔터테인먼트 걸 그룹 리센느(RESCENE)가 장기 흥행 중이다. 리센느(원이, 리브, 미나미, 메이, 제나)는 미니 1집 'SCENEDROME'(씬드롬)의 타이틀곡 'LOVE ATTACK'(러브 어택)으로 멜론 TOP100 1위 를 장기간 유지하고 있는 가운데 리메이크 싱글 'Pretty Girl'(프리티 걸)까지 2위(지난 13일 오후 11시 기준)에 오르며 두 곡을 나란히 최상위권에 올렸다. 특히 'LOVE ATTACK'은 역주행 이후 꾸준히 차트 최상위권을 지키며 리센느의 대표곡으로 자리매김했다. 'LOVE ATTACK'은 8월 멜론 월간 차트 1위까지 차지하며 뜨거운 인기를 이어오고 있고, 'Pretty Girl'(프리티 걸) 역시 8월 멜론 월간 차트 5위에 오르는 저력을 보여줬다. /사진=더뮤즈엔터테인먼트 음원 차트에서 이어진 인기

In [15]:
chat_history.add_message("\n".join(articles))
chat_history.add_user_message("리센느에 대한 반응을 알려주세요") 

# 문서 검색하고 답변을 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 
print(answer.content)

제공된 기사들을 보면 리센느에 대한 반응은 전반적으로 **매우 긍정적이며, 대중적 인지도가 빠르게 상승하고 있다**는 분위기입니다.

- **음원 차트에서 강한 반응**  
  ‘LOVE ATTACK’이 약 2년 만에 역주행해 멜론 TOP100 1위에 올랐고, 장기간 상위권을 유지했습니다. ‘Pretty Girl’도 2위까지 오르며 두 곡이 동시에 주목받았습니다.

- **대표곡과 여러 곡이 함께 흥행**  
  ‘Deja Vu’는 멜론 TOP100 5위, ‘Runaway’는 16위까지 오르는 등 특정 곡에만 관심이 집중된 것이 아니라 리센느의 여러 곡이 함께 상승세를 보였습니다.

- **음악방송 성과로 인기 확인**  
  ‘LOVE ATTACK’으로 2관왕, ‘Pretty Girl’로 4관왕을 차지해 총 6개의 음악방송 트로피를 거머쥐었습니다. 이를 바탕으로 언론에서는 리센느를 ‘5세대 대세 걸그룹’으로 평가하고 있습니다.

- **현장 반응과 인지도 상승**  
  대학 축제에서 관객들이 먼저 리센느를 알아보고 노래를 따라 부르는 등, 과거보다 팀 인지도가 크게 높아졌다는 반응이 나왔습니다.

- **멤버 개인의 화제성도 증가**  
  원이의 유튜브 콘텐츠와 미나미의 ‘야호’·갸루 캐릭터가 화제를 모으면서 팀에 대한 관심을 끌어올렸습니다. 미나미는 ‘라디오스타’ 단독 출연을 통해 예능감, 한국어 실력, 서예 실력 등 다양한 매력도 보여줬습니다.

- **광고와 방송 섭외 증가**  
  미나미가 광고 문의가 100건 이상 들어왔다고 밝힐 만큼 상업적 관심도 커졌으며, 대학 축제와 방송 출연 등 활동 범위도 넓어지고 있습니다.

다만 제공된 자료는 주로 소속사와 언론 보도 중심이므로, 구체적인 팬들의 다양한 의견이나 비판적 반응까지 확인할 수 있는 내용은 제한적입니다. 종합하면 현재 리센느는 **역주행을 계기로 음원·방송·축제·예능 전반에서 주목받는 상승세의 걸그룹**으로 받아들여지고 있습니다.
